# Commodity Delivery Gap Analysis
## Oil Tanker Imports by US PADD + Derivative Commodities (Helium, NGL)

**Data Sources:**
- **EIA API v2** — Crude oil imports by PADD/port, petroleum movements by tanker/barge, weekly stocks, STEO forecasts
- **USGS Mineral Commodity Summaries** — Helium production & supply (byproduct of natural gas processing)
- **AISstream.io** — Free real-time AIS vessel tracking (tanker positions, ETAs) — websocket API

**Thesis:** Helium has no substitute and is extracted as a byproduct of natural gas processing. Natural gas import/production disruptions are a *leading indicator* for helium supply gaps. Oil tanker delivery gaps by PADD reveal regional refinery utilization risk.

---
### Setup
Set `USE_LIVE_API = True` and add your EIA API key to pull real data.  
Otherwise, the notebook runs on synthetic data shaped to match EIA schema.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ---------- CONFIG ----------
USE_LIVE_API = False  # Flip to True when you have an EIA key
EIA_API_KEY = "YOUR_EIA_KEY_HERE"  # Register free: https://www.eia.gov/opendata/register.php

# PADDs (Petroleum Administration for Defense Districts)
PADDS = {
    "PADD 1": "East Coast",
    "PADD 2": "Midwest",
    "PADD 3": "Gulf Coast",
    "PADD 4": "Rocky Mountain",
    "PADD 5": "West Coast",
}

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

---
## 1. EIA API v2 — Pull Crude Oil Imports by PADD (Tanker/Barge)

Key endpoints:
- `petroleum/move/imp/data` — imports by PADD, mode (tanker, pipeline, barge, rail)
- `crude-oil-imports/data` — crude imports by port, country, grade
- `petroleum/stoc/wstk/data` — weekly stocks by PADD
- `steo` — Short Term Energy Outlook (forward projections)

In [ ]:
def fetch_eia_data(route, params, api_key):
    """Generic EIA API v2 fetcher. Returns DataFrame."""
    import requests
    base = "https://api.eia.gov/v2"
    url = f"{base}/{route}/data/"
    params["api_key"] = api_key
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()
    if "response" in data and "data" in data["response"]:
        return pd.DataFrame(data["response"]["data"])
    raise ValueError(f"Unexpected response structure: {list(data.keys())}")


def fetch_crude_imports_by_padd(api_key, start="2022-01", end="2026-03"):
    """Pull monthly crude oil imports by PADD via tanker/barge."""
    params = {
        "frequency": "monthly",
        "data[0]": "value",
        "facets[duoarea][]": ["R10", "R20", "R30", "R40", "R50"],  # PADDs 1-5
        "start": start,
        "end": end,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "length": 5000,
    }
    return fetch_eia_data("petroleum/move/imp", params, api_key)


def fetch_weekly_stocks(api_key, start="2024-01"):
    """Pull weekly crude oil stocks by PADD."""
    params = {
        "frequency": "weekly",
        "data[0]": "value",
        "start": start,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "length": 5000,
    }
    return fetch_eia_data("petroleum/stoc/wstk", params, api_key)


def fetch_steo_projections(api_key):
    """Pull Short Term Energy Outlook — crude oil production & imports forecasts."""
    series_ids = ["PAPR_WORLD", "COIMPUS", "COPS_SPR", "COSTPUS"]
    params = {
        "frequency": "monthly",
        "data[0]": "value",
        "facets[seriesId][]": series_ids,
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "length": 5000,
    }
    return fetch_eia_data("steo", params, api_key)


print("EIA API functions defined.")
print(f"Live mode: {USE_LIVE_API}")

---
## 2. Synthetic Data Generator (mirrors EIA schema)
When `USE_LIVE_API = False`, we generate realistic data shaped to EIA conventions.

In [ ]:
def generate_synthetic_imports():
    """Generate synthetic monthly crude imports by PADD (thousand barrels).
    Shapes based on actual EIA magnitudes:
      PADD 3 (Gulf Coast) ~65% of imports
      PADD 1 (East Coast) ~20%
      PADD 5 (West Coast) ~10%
      PADD 2 (Midwest) ~4%
      PADD 4 (Rocky Mtn) ~1%
    """
    np.random.seed(42)
    dates = pd.date_range("2022-01-01", "2026-03-01", freq="MS")
    padd_baselines = {
        "PADD 1": 40000, "PADD 2": 8000, "PADD 3": 130000,
        "PADD 4": 2000, "PADD 5": 20000
    }
    rows = []
    for padd, base in padd_baselines.items():
        # Seasonal pattern + trend + noise + a deliberate recent dip
        seasonal = base * 0.08 * np.sin(2 * np.pi * np.arange(len(dates)) / 12)
        trend = np.linspace(0, base * 0.05, len(dates))
        noise = np.random.normal(0, base * 0.06, len(dates))
        
        # Inject a delivery gap in late 2025 / early 2026 (simulating disruption)
        gap_mask = (dates >= "2025-10-01") & (dates <= "2026-02-01")
        disruption = np.where(gap_mask, -base * 0.25, 0)
        
        values = base + seasonal + trend + noise + disruption
        values = np.maximum(values, 0)
        
        for d, v in zip(dates, values):
            rows.append({
                "period": d.strftime("%Y-%m"),
                "duoarea": padd,
                "duoarea-name": PADDS[padd],
                "product": "EPC0",
                "product-name": "Crude Oil",
                "process": "IM0",
                "process-name": "Imports",
                "value": round(v, 0),
                "units": "MBBL",
            })
    return pd.DataFrame(rows)


def generate_synthetic_stocks():
    """Generate synthetic weekly crude stocks by PADD (thousand barrels)."""
    np.random.seed(99)
    dates = pd.date_range("2024-01-05", "2026-03-21", freq="W-FRI")
    padd_stock_base = {
        "PADD 1": 12000, "PADD 2": 95000, "PADD 3": 270000,
        "PADD 4": 22000, "PADD 5": 50000
    }
    rows = []
    for padd, base in padd_stock_base.items():
        level = base
        for d in dates:
            draw = np.random.normal(0, base * 0.01)
            # Draw down stocks sharply in disruption window
            if d >= pd.Timestamp("2025-10-01") and d <= pd.Timestamp("2026-02-01"):
                draw -= base * 0.008
            level = max(level + draw, base * 0.6)
            rows.append({
                "period": d.strftime("%Y-%m-%d"),
                "duoarea": padd,
                "duoarea-name": PADDS[padd],
                "value": round(level, 0),
                "units": "MBBL"
            })
    return pd.DataFrame(rows)


def generate_synthetic_helium():
    """Generate annual helium supply/demand data (million cubic meters).
    Based on USGS Mineral Commodity Summaries structure.
    US production ~40 Mcm, global ~180 Mcm. Demand growing ~2.5%/yr.
    """
    years = list(range(2018, 2027))
    us_prod = [56, 55, 50, 48, 52, 54, 56, 53, 50]  # declining trend
    world_prod = [160, 158, 150, 155, 170, 175, 180, 172, 168]
    demand = [155, 160, 148, 153, 165, 172, 180, 185, 190]  # growing
    price_per_mcf = [7.5, 8.0, 10.0, 19.0, 20.5, 22.0, 35.0, 40.0, 42.0]  # $/Mcf (Bureau of Land Management grade A)
    
    return pd.DataFrame({
        "year": years,
        "us_production_Mcm": us_prod,
        "world_production_Mcm": world_prod,
        "world_demand_Mcm": demand,
        "supply_gap_Mcm": [w - d for w, d in zip(world_prod, demand)],
        "blm_price_usd_per_mcf": price_per_mcf,
    })


def generate_synthetic_natgas_imports():
    """Generate monthly natural gas imports by pipeline + LNG (Bcf).
    Helium co-production proxy — disruptions here cascade to helium supply.
    """
    np.random.seed(77)
    dates = pd.date_range("2022-01-01", "2026-03-01", freq="MS")
    rows = []
    for mode, base in [("Pipeline", 250), ("LNG", 35)]:
        seasonal = base * 0.15 * np.sin(2 * np.pi * np.arange(len(dates)) / 12 + np.pi)
        trend = np.linspace(0, base * 0.08, len(dates))
        noise = np.random.normal(0, base * 0.04, len(dates))
        gap = np.where(
            (dates >= pd.Timestamp("2025-11-01")) & (dates <= pd.Timestamp("2026-01-01")),
            -base * 0.20, 0
        )
        values = base + seasonal + trend + noise + gap
        for d, v in zip(dates, values):
            rows.append({
                "period": d.strftime("%Y-%m"),
                "mode": mode,
                "value_bcf": round(max(v, 0), 1),
            })
    return pd.DataFrame(rows)


print("Synthetic generators ready.")

---
## 3. Load Data

In [ ]:
if USE_LIVE_API:
    print("Pulling from EIA API v2...")
    df_imports = fetch_crude_imports_by_padd(EIA_API_KEY)
    df_stocks = fetch_weekly_stocks(EIA_API_KEY)
    # Note: STEO has forward projections — key for gap forecasting
    df_steo = fetch_steo_projections(EIA_API_KEY)
    print(f"Imports: {len(df_imports)} rows | Stocks: {len(df_stocks)} rows | STEO: {len(df_steo)} rows")
else:
    print("Using synthetic data (set USE_LIVE_API=True for real data)")
    df_imports = generate_synthetic_imports()
    df_stocks = generate_synthetic_stocks()

df_helium = generate_synthetic_helium()  # USGS data is annual PDFs, always synthetic here
df_natgas = generate_synthetic_natgas_imports()

# Parse dates
df_imports["date"] = pd.to_datetime(df_imports["period"])
df_stocks["date"] = pd.to_datetime(df_stocks["period"])
df_natgas["date"] = pd.to_datetime(df_natgas["period"])
df_imports["value"] = pd.to_numeric(df_imports["value"], errors="coerce")
df_stocks["value"] = pd.to_numeric(df_stocks["value"], errors="coerce")

print(f"\nImports shape: {df_imports.shape}")
print(f"Stocks shape:  {df_stocks.shape}")
print(f"NatGas shape:  {df_natgas.shape}")
print(f"Helium shape:  {df_helium.shape}")
df_imports.head(3)

---
## 4. Crude Oil Import Gap Analysis by PADD

**Method:** Compare trailing 12-month average to recent months. A "gap" = actual imports below the trailing norm by > 1 std dev.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14), sharex=True)
axes = axes.flatten()

gap_summary = []

for i, (padd, name) in enumerate(PADDS.items()):
    ax = axes[i]
    sub = df_imports[df_imports["duoarea"] == padd].sort_values("date").copy()
    
    # Trailing 12-month stats
    sub["ma12"] = sub["value"].rolling(12, min_periods=6).mean()
    sub["std12"] = sub["value"].rolling(12, min_periods=6).std()
    sub["lower_band"] = sub["ma12"] - sub["std12"]
    sub["gap"] = sub["value"] - sub["ma12"]
    sub["in_gap"] = sub["value"] < sub["lower_band"]
    
    # Plot
    ax.plot(sub["date"], sub["value"], 'o-', ms=3, label="Actual imports", color="#2563eb")
    ax.plot(sub["date"], sub["ma12"], '--', label="12-mo avg", color="#64748b", lw=2)
    ax.fill_between(sub["date"], sub["lower_band"], sub["ma12"],
                     alpha=0.15, color="#64748b", label="Normal band")
    
    # Highlight gap periods
    gap_dates = sub[sub["in_gap"]]["date"]
    for gd in gap_dates:
        ax.axvspan(gd - timedelta(days=15), gd + timedelta(days=15),
                   alpha=0.25, color="#ef4444")
    
    ax.set_title(f"{padd}: {name}", fontweight="bold")
    ax.set_ylabel("Thousand Barrels")
    ax.legend(fontsize=8, loc="lower left")
    
    # Summary stats for recent gap
    recent = sub[sub["date"] >= "2025-09-01"]
    if recent["in_gap"].any():
        gap_months = recent[recent["in_gap"]]["date"].dt.strftime("%Y-%m").tolist()
        avg_deficit = recent[recent["in_gap"]]["gap"].mean()
        gap_summary.append({
            "padd": padd, "region": name,
            "gap_months": ", ".join(gap_months),
            "avg_deficit_mbbl": round(avg_deficit, 0),
        })

# Hide unused subplot
axes[5].set_visible(False)
fig.suptitle("Crude Oil Imports by PADD — Delivery Gap Detection", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

if gap_summary:
    print("\n=== DETECTED DELIVERY GAPS (recent) ===")
    display(pd.DataFrame(gap_summary))
else:
    print("No significant delivery gaps detected in recent window.")

---
## 5. Weekly Stocks Drawdown — Days-of-Supply Risk

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=False)
axes = axes.flatten()

for i, (padd, name) in enumerate(PADDS.items()):
    ax = axes[i]
    sub = df_stocks[df_stocks["duoarea"] == padd].sort_values("date").copy()
    
    # Rate of change (4-week)
    sub["delta_4w"] = sub["value"].diff(4)
    sub["pct_change_4w"] = sub["value"].pct_change(4) * 100
    
    color = sub["delta_4w"].apply(lambda x: "#ef4444" if x < 0 else "#22c55e")
    ax.bar(sub["date"], sub["delta_4w"], width=5, color=color, alpha=0.7)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(f"{padd}: {name}\n4-Week Stock Change", fontweight="bold")
    ax.set_ylabel("MBBL change")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis='x', rotation=45)

axes[5].set_visible(False)
fig.suptitle("Weekly Crude Stock Changes by PADD — Drawdown Alert", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 6. Natural Gas Imports — Helium Supply Leading Indicator

Helium is extracted during natural gas processing (cryogenic separation).  
**No natural gas throughput → no helium production.** Pipeline and LNG disruptions cascade directly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for mode, color, ax in [("Pipeline", "#2563eb", ax1), ("LNG", "#7c3aed", ax2)]:
    sub = df_natgas[df_natgas["mode"] == mode].sort_values("date")
    sub["ma6"] = sub["value_bcf"].rolling(6, min_periods=3).mean()
    sub["std6"] = sub["value_bcf"].rolling(6, min_periods=3).std()
    
    ax.plot(sub["date"], sub["value_bcf"], 'o-', ms=3, color=color, label=f"{mode} imports")
    ax.plot(sub["date"], sub["ma6"], '--', color="#64748b", label="6-mo avg")
    ax.fill_between(sub["date"], sub["ma6"] - sub["std6"], sub["ma6"] + sub["std6"],
                     alpha=0.1, color="#64748b")
    
    # Flag disruptions
    gap_mask = sub["value_bcf"] < (sub["ma6"] - 1.5 * sub["std6"])
    ax.scatter(sub[gap_mask]["date"], sub[gap_mask]["value_bcf"],
               color="#ef4444", s=80, zorder=5, label="Disruption", marker="v")
    
    ax.set_title(f"Natural Gas Imports — {mode}", fontweight="bold")
    ax.set_ylabel("Billion Cubic Feet (Bcf)")
    ax.legend(fontsize=9)

fig.suptitle("Natural Gas Import Disruptions → Helium Supply Chain Risk",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Helium Supply-Demand Gap + Price Signal

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Supply vs Demand
x = df_helium["year"]
ax1.bar(x - 0.2, df_helium["world_production_Mcm"], 0.35, label="World Production", color="#2563eb")
ax1.bar(x + 0.2, df_helium["world_demand_Mcm"], 0.35, label="World Demand", color="#f97316")
ax1.axhline(0, color="black", lw=0.5)

# Shade deficit years
for _, row in df_helium[df_helium["supply_gap_Mcm"] < 0].iterrows():
    ax1.axvspan(row["year"] - 0.45, row["year"] + 0.45, alpha=0.15, color="#ef4444")

ax1.set_title("Global Helium Supply vs. Demand", fontweight="bold")
ax1.set_ylabel("Million Cubic Meters")
ax1.legend()
ax1.set_xticks(x)

# Price trajectory
ax2.plot(x, df_helium["blm_price_usd_per_mcf"], 'o-', color="#dc2626", lw=2, ms=7)
ax2.fill_between(x, df_helium["blm_price_usd_per_mcf"], alpha=0.1, color="#dc2626")
ax2.set_title("Helium Price (BLM Grade-A)", fontweight="bold")
ax2.set_ylabel("USD per Mcf")
ax2.set_xticks(x)

# Annotate the inflection
ax2.annotate("Shortage pricing\nkicks in", xy=(2023, 35),
             xytext=(2021, 38), fontsize=10,
             arrowprops=dict(arrowstyle="->", color="#dc2626"),
             color="#dc2626", fontweight="bold")

fig.suptitle("Helium: No Substitutes, Constrained Supply, Surging Demand (MRI, Semiconductors, Quantum)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\n=== Helium Supply Gap Summary ===")
display(df_helium[["year", "supply_gap_Mcm", "blm_price_usd_per_mcf"]].tail(5))

---
## 8. Composite Gap Scorecard

Combine oil, natural gas, and helium signals into a single delivery-risk view.

In [ ]:
def compute_gap_score(series, window=6):
    """Z-score relative to trailing window. Negative = below-normal deliveries."""
    ma = series.rolling(window, min_periods=3).mean()
    std = series.rolling(window, min_periods=3).std()
    return (series - ma) / std.replace(0, np.nan)


# Monthly crude imports — national total
national_imports = df_imports.groupby("date")["value"].sum().sort_index()
oil_z = compute_gap_score(national_imports, 12)

# Natural gas total
natgas_total = df_natgas.groupby("date")["value_bcf"].sum().sort_index()
gas_z = compute_gap_score(natgas_total, 6)

# Align on common monthly index
common_idx = oil_z.index.intersection(gas_z.index)
scorecard = pd.DataFrame({
    "oil_import_z": oil_z.loc[common_idx],
    "natgas_import_z": gas_z.loc[common_idx],
}).dropna()

# Composite score (equal weight)
scorecard["composite_gap_score"] = (scorecard["oil_import_z"] + scorecard["natgas_import_z"]) / 2

# Plot
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(scorecard.index, scorecard["oil_import_z"], label="Oil Import Z", color="#2563eb", lw=1.5)
ax.plot(scorecard.index, scorecard["natgas_import_z"], label="NatGas Import Z", color="#7c3aed", lw=1.5)
ax.plot(scorecard.index, scorecard["composite_gap_score"], label="Composite Gap Score",
        color="#dc2626", lw=2.5)
ax.axhline(-1, color="#f97316", ls="--", label="Warning threshold (-1σ)")
ax.axhline(-2, color="#ef4444", ls="--", label="Critical threshold (-2σ)")
ax.axhline(0, color="black", lw=0.5)

# Shade critical zones
ax.fill_between(scorecard.index, -2, scorecard["composite_gap_score"],
                where=scorecard["composite_gap_score"] < -2,
                alpha=0.3, color="#ef4444", interpolate=True)

ax.set_title("Composite Delivery Gap Scorecard — Oil + Natural Gas (Helium Proxy)",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Z-Score (negative = below normal)")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

# Recent alert table
alerts = scorecard[scorecard["composite_gap_score"] < -1].tail(10)
if not alerts.empty:
    print("\n=== MONTHS WITH GAP ALERTS (composite < -1σ) ===")
    display(alerts.round(2))

---
## 9. AIS Vessel Tracking — Real-Time Tanker Watch

**AISstream.io** provides free websocket-based AIS data. Below is the connection pattern to track tankers bound for US ports. Run this cell in a local environment with network access.

Ship type codes: `80-89` = Tankers

In [ ]:
# =====================================================
# AISstream.io — Real-time tanker tracking (reference)
# Register at https://aisstream.io to get an API key
# =====================================================

AISSTREAM_CODE = '''
import asyncio
import websockets
import json

AISSTREAM_API_KEY = "YOUR_AISSTREAM_KEY"

# Bounding boxes for major US port regions
US_PORT_BOXES = [
    [[24.5, -98.0], [30.5, -87.0]],   # Gulf Coast (Houston, LOOP, New Orleans)
    [[37.5, -76.5], [40.7, -73.5]],   # East Coast (NY/NJ, Philadelphia, Delaware)
    [[33.5, -119.0], [34.5, -117.5]], # West Coast (Los Angeles/Long Beach)
    [[47.0, -123.0], [48.5, -122.0]], # Pacific NW (Puget Sound)
]

async def track_tankers():
    async with websockets.connect("wss://stream.aisstream.io/v0/stream") as ws:
        subscribe = {
            "APIKey": AISSTREAM_API_KEY,
            "BoundingBoxes": US_PORT_BOXES,
            "FilterShipType": list(range(80, 90)),  # Tankers only
        }
        await ws.send(json.dumps(subscribe))
        
        tanker_log = []
        async for msg in ws:
            data = json.loads(msg)
            msg_type = data.get("MessageType", "")
            
            if msg_type == "PositionReport":
                pos = data["Message"]["PositionReport"]
                meta = data.get("MetaData", {})
                entry = {
                    "mmsi": meta.get("MMSI"),
                    "name": meta.get("ShipName", "").strip(),
                    "lat": pos.get("Latitude"),
                    "lon": pos.get("Longitude"),
                    "speed_kn": pos.get("Sog"),
                    "heading": pos.get("TrueHeading"),
                    "destination": meta.get("Destination", "").strip(),
                    "eta": meta.get("ETA", ""),
                    "timestamp": meta.get("time_utc"),
                }
                tanker_log.append(entry)
                
                if len(tanker_log) % 50 == 0:
                    print(f"Tracked {len(tanker_log)} tanker positions...")
                    # Could write to DataFrame, SQLite, etc.
            
            if len(tanker_log) >= 500:  # Demo cap
                break
        
        return pd.DataFrame(tanker_log)

# Run: df_tankers = asyncio.run(track_tankers())
'''

print("AISstream.io tanker tracking code (reference — run locally with API key):")
print(AISSTREAM_CODE)

---
## 10. Trackable Derivative Commodities — Data Source Map

Commodities whose supply chains are measurable via shipping/import data and have tradeable derivatives.

In [ ]:
commodity_map = pd.DataFrame([
    {"commodity": "Crude Oil",
     "derivative_instruments": "WTI/Brent futures, crack spreads, PADD basis",
     "public_data_source": "EIA API v2 (petroleum/move/imp, crude-oil-imports)",
     "shipping_trackable": True,
     "lead_time_visibility": "2-6 weeks (AIS tanker ETA)",
     "gap_signal": "PADD import shortfall vs 12-mo avg"},
    
    {"commodity": "Helium",
     "derivative_instruments": "No direct futures; equities (DME, RLPI, PHX), procurement contracts",
     "public_data_source": "USGS Mineral Commodity Summaries (annual); BLM auction prices",
     "shipping_trackable": False,
     "lead_time_visibility": "Lagging (annual data); proxy via NatGas throughput",
     "gap_signal": "NatGas processing disruptions → helium co-production collapse"},
    
    {"commodity": "Natural Gas (LNG)",
     "derivative_instruments": "Henry Hub futures, JKM (Asia LNG), TTF (Europe)",
     "public_data_source": "EIA API v2 (natural-gas/move), FERC LNG reports",
     "shipping_trackable": True,
     "lead_time_visibility": "2-4 weeks (LNG carrier AIS)",
     "gap_signal": "LNG import volume vs seasonal norm"},
    
    {"commodity": "Gasoline / Distillates",
     "derivative_instruments": "RBOB gasoline futures, heating oil futures, crack spreads",
     "public_data_source": "EIA API v2 (petroleum/move/wkly, petroleum/stoc/wstk)",
     "shipping_trackable": True,
     "lead_time_visibility": "1-3 weeks (product tanker AIS)",
     "gap_signal": "Weekly stocks drawdown rate by PADD"},
    
    {"commodity": "Propane / NGLs",
     "derivative_instruments": "Mt. Belvieu propane, ethane/butane swaps",
     "public_data_source": "EIA API v2 (petroleum/stoc, natural-gas/move)",
     "shipping_trackable": True,
     "lead_time_visibility": "1-2 weeks",
     "gap_signal": "NGL stock levels + fractionation utilization"},
    
    {"commodity": "Lithium",
     "derivative_instruments": "CME lithium hydroxide futures, equity (ALB, SQM, LTHM)",
     "public_data_source": "USGS MCS, Fastmarkets, Benchmark Minerals",
     "shipping_trackable": True,
     "lead_time_visibility": "4-8 weeks (bulk carrier AIS from Chile/Australia)",
     "gap_signal": "Chilean/Australian port departures vs contract volumes"},
])

# Pretty display
display(commodity_map.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold')]}
]))

---
## 11. Next Steps — Making This Live

1. **Register for EIA API key** (free): https://www.eia.gov/opendata/register.php  
   Set `USE_LIVE_API = True` and paste your key above.

2. **Register for AISstream.io** (free): https://aisstream.io  
   Run the websocket code in Cell 9 to collect real-time tanker positions + ETAs.

3. **USGS Helium data**: Download the latest Mineral Commodity Summary from  
   https://www.usgs.gov/centers/national-minerals-information-center/helium-statistics-and-information  
   or use the data release at https://data.usgs.gov/datacatalog/data/USGS:6797fe71d34ea8c18376e1af

4. **Scheduling**: Wrap the EIA + AIS pulls in a cron job or Airflow DAG to build a time series DB.  
   The gap scorecard (Section 8) can fire Slack/email alerts when composite < -1σ.

5. **Derivative overlay**: Pull futures curves from CME/ICE via `yfinance` (`CL=F`, `NG=F`, `RB=F`)  
   to overlay physical delivery gaps with market pricing.